# 第2回 宿題（発展）: 用語集で LoRA 学習し、用語集の検索と比べる
- 講座の用語集（data/glossary.json、58語）から Q&A を作り、Qwen3.5-0.8B を LoRA で学習する
- 評価は3種類: ①学習に使った言い回し ②学習に使っていない言い回し ③用語集にない語（「載っていません」と答えられるか）
- 工夫: 用語集にない語への「載っていません」の例も学習に混ぜる（作り話を減らすため）。評価に使う「載っていない語」は学習に使わない
- 比較: 学習なし（before）／学習あり（after）／用語集を検索して定義を返す仕組み（モデルなし）
- ランタイム: T4 GPU。上から順に実行する（20分ほど）

In [ ]:
# [0] ライブラリ（第2回演習ノートと同じ版）と講座リポジトリ
%pip uninstall -y -q torchao
%pip install -q "transformers==5.18.0" "peft==0.21.1" "trl==1.14.2" "accelerate==1.15.0" "datasets==5.1.0"
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/matsuolab/lecture-ai-engineering-2026.git /content/repo
!ls /content/repo/data

In [ ]:
# [1] 学習データと評価データを作る
import json, random, unicodedata, time
random.seed(42)
TERMS = json.load(open("/content/repo/data/glossary.json", encoding="utf-8"))["terms"]
SYSTEM = ("あなたはAIエンジニアリング講座の用語集の回答係です。用語集に載っている用語は、その定義を答えてください。"
          "用語集に載っていない用語は「この用語は用語集に載っていません。」とだけ答えてください。")
REFUSAL = "この用語は用語集に載っていません。"
TRAIN_T = ["「{t}」とは何ですか？", "{t}について簡単に説明してください。", "{t}の意味を教えてください。", "講座で出てきた{t}って何？"]
PARA_T = ["{t}を一言でいうと何ですか。", "初心者向けに{t}を説明して。"]   # 学習には使わない言い回し

def norm(s):
    return unicodedata.normalize("NFKC", s).lower().replace(" ", "")

NAMES = {norm(n) for t in TERMS for n in [t["term"]] + t.get("aliases", [])}
CANDIDATES = ["量子アニーリング", "ブロックチェーン", "遺伝的アルゴリズム", "サポートベクターマシン", "ベイズ最適化",
              "ナレッジグラフ", "エッジコンピューティング", "連合学習", "敵対的生成ネットワーク", "ランダムフォレスト",
              "主成分分析", "モンテカルロ木探索", "変分オートエンコーダ", "ニューロモーフィックチップ", "デジタルツイン",
              "差分プライバシー", "グラフニューラルネットワーク", "カルマンフィルタ", "k近傍法", "決定木", "隠れマルコフモデル",
              "フーリエ変換", "勾配ブースティング", "自己組織化マップ"]
# 用語集の名前と重なるもの（部分一致も）は除く
UNKNOWN = [c for c in CANDIDATES if not any(norm(c) in n or n in norm(c) for n in NAMES)]
random.shuffle(UNKNOWN)
UNK_TRAIN, UNK_TEST = UNKNOWN[: len(UNKNOWN) // 2], UNKNOWN[len(UNKNOWN) // 2:]

def msgs(q, a=None):
    m = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    return m + ([{"role": "assistant", "content": a}] if a is not None else [])

train = []
for t in TERMS:
    names = [t["term"]] + t.get("aliases", [])[:1]
    for i, tpl in enumerate(TRAIN_T):
        name = names[i % len(names)] if i == 3 else t["term"]   # 4つ目の言い回しは別名で
        train.append({"messages": msgs(tpl.format(t=name), t["definition"])})
for u in UNK_TRAIN:
    for tpl in TRAIN_T[:2]:
        train.append({"messages": msgs(tpl.format(t=u), REFUSAL)})
random.shuffle(train)

EVAL = {
    "seen": [{"q": TRAIN_T[0].format(t=t["term"]), "gold": t["definition"], "id": t["id"]} for t in TERMS],
    "para": [{"q": PARA_T[i % 2].format(t=t["term"]), "gold": t["definition"], "id": t["id"]} for i, t in enumerate(TERMS)],
    "unknown": [{"q": PARA_T[0].format(t=u), "gold": None, "id": u} for u in UNK_TEST],
}
print("用語", len(TERMS), "/ 学習例", len(train), "/ 載っていない語: 学習", UNK_TRAIN, "評価", UNK_TEST)
print({k: len(v) for k, v in EVAL.items()})

In [ ]:
# [2] モデルの読み込みと生成・採点の関数
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL, REV = "Qwen/Qwen3.5-0.8B", "2fc06364715b967f1860aea9cf38778875588b17"
tok = AutoTokenizer.from_pretrained(MODEL, revision=REV)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, dtype=torch.float32).to("cuda")

@torch.no_grad()
def generate(m, questions, bs=8, max_new_tokens=160):
    m.eval()
    outs = []
    for i in range(0, len(questions), bs):
        prompts = [tok.apply_chat_template(msgs(q), tokenize=False, add_generation_prompt=True, enable_thinking=False)
                   for q in questions[i:i + bs]]
        enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
        gen = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for g in gen]
    return outs

def bigrams(s):
    s = norm(s)
    return [s[i:i + 2] for i in range(len(s) - 1)]

def char_f1(pred, gold):
    # 文字2-gramの重なりによる F1（定義文との近さの目安）
    p, g = bigrams(pred), bigrams(gold)
    if not p or not g:
        return 0.0
    from collections import Counter
    c = sum((Counter(p) & Counter(g)).values())
    if c == 0:
        return 0.0
    prec, rec = c / len(p), c / len(g)
    return 2 * prec * rec / (prec + rec)

def is_refusal(s):
    return "載っていません" in s

def score(outputs):
    res = {}
    for k, items in EVAL.items():
        outs = outputs[k]
        if k == "unknown":
            res[k] = {"refusal_rate": round(sum(map(is_refusal, outs)) / len(outs), 3)}
        else:
            f1 = [char_f1(o, it["gold"]) for o, it in zip(outs, items)]
            res[k] = {"char_f1": round(sum(f1) / len(f1), 3),
                      "f1>=0.5の割合": round(sum(x >= 0.5 for x in f1) / len(f1), 3),
                      "誤って拒否": round(sum(map(is_refusal, outs)) / len(outs), 3)}
    return res

def run_eval(m, label):
    t0 = time.time()
    outs = {k: generate(m, [it["q"] for it in v]) for k, v in EVAL.items()}
    r = score(outs)
    r["seconds"] = round(time.time() - t0, 1)
    print(label, json.dumps(r, ensure_ascii=False))
    return r, outs

In [ ]:
# [3] 学習前（before）
before, before_outs = run_eval(model, "before")
for k in EVAL:
    print("--", k, "| 質問:", EVAL[k][0]["q"], "\n   出力:", before_outs[k][0][:120])

In [ ]:
# [4] 用語集の検索（モデルなし）: 質問に含まれる用語名で引き、なければ文字2-gramの近さで引く。近さが低ければ「載っていません」
from collections import Counter
INDEX = [(t, [norm(n) for n in [t["term"]] + t.get("aliases", [])]) for t in TERMS]

def lookup(q, threshold=0.25):
    nq = norm(q)
    hits = [(len(n), t) for t, names in INDEX for n in names if n in nq]
    if hits:
        return max(hits, key=lambda x: x[0])[1]["definition"]
    def sim(t):
        a, b = Counter(bigrams(q)), Counter(bigrams(t["term"] + "".join(t.get("aliases", []))))
        c = sum((a & b).values())
        return c / max(1, sum(b.values()))
    best = max(TERMS, key=sim)
    return best["definition"] if sim(best) >= threshold else REFUSAL

t0 = time.time()
search_outs = {k: [lookup(it["q"]) for it in v] for k, v in EVAL.items()}
search = score(search_outs)
search["seconds"] = round(time.time() - t0, 3)
print("search", json.dumps(search, ensure_ascii=False))

In [ ]:
# [5] LoRA 学習（第2回演習と同じ設定を基本に、データだけ用語集に）
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
CONFIG = {"epochs": 3, "lr": 2e-4, "lora_r": 16, "lora_alpha": 32, "batch": 8, "seed": 42}
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir="/content/glossary_ckpt", num_train_epochs=CONFIG["epochs"], per_device_train_batch_size=CONFIG["batch"],
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=5, logging_steps=5, save_strategy="no",
                max_length=512, packing=False, report_to="none", seed=CONFIG["seed"])
model.config.use_cache = False
trainer = SFTTrainer(model=model, args=sft, train_dataset=Dataset.from_list(train), peft_config=lora, processing_class=tok)
t0 = time.time()
trainer.train()
TRAIN_SEC = round(time.time() - t0, 1)
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print("学習時間", TRAIN_SEC, "秒 / ステップ", trainer.state.global_step, "/ loss", [round(x, 3) for x in losses])

In [ ]:
# [6] 学習後（after）と比較表
tuned = trainer.model
tuned.config.use_cache = True
after, after_outs = run_eval(tuned, "after")
import pandas as pd
rows = []
for name, r in [("学習前", before), ("学習後(LoRA)", after), ("用語集の検索", search)]:
    rows.append({"方式": name, "①既出の言い回し F1": r["seen"]["char_f1"], "②新しい言い回し F1": r["para"]["char_f1"],
                 "②でF1≥0.5": r["para"]["f1>=0.5の割合"], "①②で誤って拒否": round((r["seen"]["誤って拒否"] + r["para"]["誤って拒否"]) / 2, 3),
                 "③載っていない語を拒否": r["unknown"]["refusal_rate"], "評価の時間(秒)": r["seconds"]})
display(pd.DataFrame(rows))
for k in EVAL:
    print("--", k, "| 質問:", EVAL[k][1]["q"], "\n   学習後:", after_outs[k][1][:150])

In [ ]:
# [7] 結果を保存して、この出力を貼る（answer.md に使う）
result = {"config": CONFIG, "n_train": len(train), "unknown_train": UNK_TRAIN, "unknown_test": UNK_TEST,
          "train_seconds": TRAIN_SEC, "final_loss": losses[-1] if losses else None,
          "before": before, "after": after, "search": search,
          "examples": {k: {"q": EVAL[k][1]["q"], "before": before_outs[k][1][:200], "after": after_outs[k][1][:200],
                           "search": search_outs[k][1][:200]} for k in EVAL}}
json.dump(result, open("/content/result_advanced.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(result, ensure_ascii=False, indent=1))